# GCSI Quickstart

## Global Commodity Sentiment Indices: first call, first chart

This notebook gets you from an API key to a chart of Brent Crude sentiment in a few minutes. It:

1. Authenticates and pulls a week of the **hourly index** for one commodity (`BZ_COM`)
2. Shows the schema and what each field means
3. Turns hourly bars into a daily series for each **horizon** (`short_term`, `medium_term`, `long_term`)
4. Lists the topics that carried the most news this week

| Key fact | Value |
|---|---|
| Granularity | One row per ticker × topic × index type × horizon × hour |
| `extraction_time` | **End** of the hour (UTC), binned by when Permutable received the headline |
| Sign | Positive = bullish for the commodity price |
| Missing hours | Omitted, not zero-filled |

> **Disclaimer:** This notebook is provided for informational and research purposes only. Nothing here constitutes financial advice or a recommendation to buy, sell, or hold any asset. Sentiment scores are model outputs describing how news was judged at publication; they are not price forecasts.

In [ ]:
%pip install requests pandas numpy matplotlib

## Step 1: Configuration

Set `DATA_SOURCE = 'zip'` to run from a bulk download instead of the API. Point `ZIP_DIR` at the folder
holding per-ticker ZIPs (for example `BZ_COM.zip`) from the Datasets page.

In [ ]:
import getpass
import os
import random
import time
import zipfile
from datetime import date, datetime, timedelta, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests


def load_env_file() -> None:
    """Read KEY=VALUE pairs from the nearest .env file without overriding variables already set."""
    for base in [Path.cwd(), *Path.cwd().parents]:
        env_path = base / '.env'
        if env_path.exists():
            for line in env_path.read_text().splitlines():
                line = line.strip()
                if line and not line.startswith('#') and '=' in line:
                    key, value = line.split('=', 1)
                    os.environ.setdefault(key.strip(), value.strip().strip('"'))
            return


def get_api_key() -> str:
    load_env_file()
    key = os.environ.get('API_KEY') or os.environ.get('PERMUTABLE_API_KEY')
    if not key:
        key = getpass.getpass('Enter your Permutable AI API key: ')
    return key


DATA_SOURCE = 'api'          # 'api' or 'zip'
ZIP_DIR = Path('data/gcsi_zips')

TICKER = 'BZ_COM'            # Brent Crude
INDEX_TYPE = 'COMBINED'      # EXPLICIT | RELATED_ENTITY | MACROECONOMIC | COMBINED
SENTIMENT_TYPES = ['short_term', 'medium_term', 'long_term']
LOOKBACK_DAYS = 7

BASE_URL = os.environ.get('BASE_URL', 'https://copilot-api.permutable.ai/v1')

# GCSI routes. The ticker and topic are query presets, not part of the path.
INDEX_LIVE_PATH = '/headlines/index/gcsi/live'
INDEX_HISTORICAL_PATH = '/headlines/index/gcsi/historical'
FEED_LIVE_PATH = '/headlines/feed/live/gcsi'
FEED_HISTORICAL_PATH = '/headlines/feed/historical/gcsi'
PAGE_LIMIT = 1000
MAX_LOOKBACK_DAYS = 90

API_KEY = get_api_key() if DATA_SOURCE == 'api' else None
print(f'Source: {DATA_SOURCE} | ticker: {TICKER} | index type: {INDEX_TYPE} | last {LOOKBACK_DAYS} days')

## Step 2: Helpers

In [ ]:
SESSION = requests.Session()
SESSION.headers.update({'x-api-key': API_KEY, 'accept': 'application/json'})


def api_get(path: str, params: dict | None = None, max_retries: int = 5):
    """GET a Permutable API path, backing off exponentially (with jitter) on 429 and 5xx responses."""
    params = {k: v for k, v in (params or {}).items() if v is not None}
    delay = 1.0
    for attempt in range(max_retries + 1):
        response = SESSION.get(BASE_URL + path, params=params, timeout=60)
        if response.status_code == 429 or response.status_code >= 500:
            if attempt == max_retries:
                response.raise_for_status()
            time.sleep(delay + random.uniform(0, delay / 2))
            delay *= 2
            continue
        if response.status_code in (401, 403):
            raise PermissionError(
                f'{response.status_code} on {path}: check the API key, and that your licence includes GCSI.'
            )
        response.raise_for_status()
        return response.json()


def _records(body) -> list[dict]:
    """Live endpoints return a list; historical endpoints return {data, has_more, next_token}."""
    return body['data'] if isinstance(body, dict) else body


def _stamp(records: list[dict], **dims) -> list[dict]:
    """Responses can omit the dimensions you asked for, so write them back onto each record."""
    for record in records:
        for key, value in dims.items():
            if value is not None:
                record.setdefault(key, value)
    return records


def _paginate(path: str, params: dict) -> list[dict]:
    params = {**params, 'limit': PAGE_LIMIT}
    rows: list[dict] = []
    while True:
        body = api_get(path, params)
        rows.extend(_records(body))
        if not (isinstance(body, dict) and body.get('has_more')):
            return rows
        params['next_token'] = body['next_token']


TOPIC_SEPARATOR = '::'  # Sector::L1::L2, the form the API returns


def normalise_topic(labels: pd.Series) -> pd.Series:
    """Write each label as sector, driver and topic joined by TOPIC_SEPARATOR, whatever separator the source used."""
    return labels.astype('string').str.replace(r'\s*(?:::|-)\s*', TOPIC_SEPARATOR, regex=True)


INDEX_VALUES = ['headline_count', 'sentiment_sum', 'sentiment_std']


def _index_params(ticker: str, index_type: str, sentiment_type: str) -> dict:
    # One ticker per call with every topic. A sector preset or ticker_preset=ALL can exceed the 1000-row live cap.
    return {'ticker_preset': ticker, 'topic_preset': 'ALL', 'index_type': index_type, 'sentiment_type': sentiment_type}


def fetch_live_index(ticker: str, index_type: str, sentiment_type: str) -> list[dict]:
    """Most recent hourly bars for one ticker. The live route is a plain list and the newest bars can still be revised."""
    body = api_get(INDEX_LIVE_PATH, _index_params(ticker, index_type, sentiment_type))
    return _stamp(_records(body), ticker=ticker, index_type=index_type, sentiment_type=sentiment_type)


def fetch_historical_index(
    ticker: str, index_type: str, sentiment_type: str, start: date, end: date | None = None
) -> list[dict]:
    """Hourly bars between start and end (inclusive dates), following next_token until has_more is false."""
    params = _index_params(ticker, index_type, sentiment_type)
    params.update({'start_date': start.isoformat(), 'end_date': end.isoformat() if end else None})
    rows = _paginate(INDEX_HISTORICAL_PATH, params)
    return _stamp(rows, ticker=ticker, index_type=index_type, sentiment_type=sentiment_type)


def to_index_frame(records: list[dict]) -> pd.DataFrame:
    """Records → DataFrame with a UTC extraction_time and every value column present."""
    frame = pd.DataFrame(records)
    for column in ['ticker', 'topic_name', 'index_type', 'sentiment_type', 'extraction_time', *INDEX_VALUES]:
        if column not in frame:
            frame[column] = np.nan
    frame['extraction_time'] = pd.to_datetime(frame['extraction_time'], utc=True)
    frame['topic_name'] = normalise_topic(frame['topic_name'])
    return frame


def read_index_zip(
    path: Path, index_types: list[str] | None = None, sentiment_types: list[str] | None = None
) -> pd.DataFrame:
    """Read one ticker's bulk hourly-index ZIP into a long frame.

    Handles key=value folders (index_type=COMBINED/sentiment_type=short_term/part.csv) and plain topic
    folders. Dimensions found only in the folder path are added as columns, value columns missing from
    the file are added as NaN, and topic labels are written in the `::` form.
    """
    ticker = Path(path).stem
    frames = []
    with zipfile.ZipFile(path) as archive:
        for name in archive.namelist():
            if not name.endswith('.csv'):
                continue
            folders = name.split('/')[:-1]
            dims = dict(part.split('=', 1) for part in folders if '=' in part)
            plain = [part for part in folders if '=' not in part and part != ticker]
            if index_types and dims.get('index_type', index_types[0]) not in index_types:
                continue
            if sentiment_types and dims.get('sentiment_type', sentiment_types[0]) not in sentiment_types:
                continue
            with archive.open(name) as handle:
                frame = pd.read_csv(handle)
            for key, value in dims.items():
                if key not in frame:
                    frame[key] = value
            frame = frame.rename(columns={'publication_time': 'extraction_time', 'topic': 'topic_name'})
            if 'topic_name' not in frame and plain:
                frame['topic_name'] = plain[-1]
            frames.append(frame)
    if not frames:
        raise ValueError(f'{path}: no CSV files matched the requested index and sentiment types')
    frame = pd.concat(frames, ignore_index=True)
    if 'ticker' not in frame:
        frame['ticker'] = ticker
    for column in INDEX_VALUES:
        if column not in frame:
            frame[column] = np.nan
    frame['extraction_time'] = pd.to_datetime(frame['extraction_time'], utc=True)  # files carry UTC times
    frame['topic_name'] = normalise_topic(frame['topic_name'])
    if index_types:
        frame = frame[frame['index_type'].isin(index_types)]
    if sentiment_types:
        frame = frame[frame['sentiment_type'].isin(sentiment_types)]
    return frame.reset_index(drop=True)


def pool(frame: pd.DataFrame, keys: list[str]) -> pd.DataFrame:
    """Combine index rows into larger buckets (hours → days, topics → drivers, ...).

    Counts and sums add. The sample standard deviation is pooled exactly: each row's sum of squares is
    rebuilt as (n - 1)·std² + n·mean², summed, and turned back into a sample standard deviation.
    Averaging hourly standard deviations would be wrong.
    """
    n = frame['headline_count'].astype(float)
    mean = frame['sentiment_sum'] / n
    sumsq = (n - 1).clip(lower=0) * frame['sentiment_std'].fillna(0.0) ** 2 + n * mean**2
    out = (
        frame.assign(_sumsq=sumsq)
        .groupby(keys, as_index=False, observed=True)[['headline_count', 'sentiment_sum', '_sumsq']]
        .sum()
    )
    n = out['headline_count'].astype(float)
    out['sentiment_avg'] = out['sentiment_sum'] / n
    variance = (out['_sumsq'] - n * out['sentiment_avg'] ** 2) / (n - 1)
    out['sentiment_std'] = np.sqrt(variance.where(n > 1, 0.0).clip(lower=0.0))
    return out.drop(columns='_sumsq')


def receipt_date(extraction_time: pd.Series) -> pd.Series:
    """UTC date the headlines were received. A bar stamped 00:00 covers 23:00–24:00 of the previous day."""
    return (extraction_time - pd.Timedelta(hours=1)).dt.floor('D').dt.tz_localize(None)

## Step 3: Load a week of data

From the API, one historical request per horizon. From a ZIP, the last `LOOKBACK_DAYS` of the file.

In [ ]:
if DATA_SOURCE == 'api':
    start = (datetime.now(timezone.utc) - timedelta(days=LOOKBACK_DAYS)).date()
    records = []
    for sentiment_type in SENTIMENT_TYPES:
        rows = fetch_historical_index(TICKER, INDEX_TYPE, sentiment_type, start)
        print(f'{sentiment_type:>12}: {len(rows):,} hourly rows')
        records.extend(rows)
    hourly = to_index_frame(records)
else:
    hourly = read_index_zip(ZIP_DIR / f'{TICKER}.zip', [INDEX_TYPE], SENTIMENT_TYPES)
    cutoff = hourly['extraction_time'].max() - pd.Timedelta(days=LOOKBACK_DAYS)
    hourly = hourly[hourly['extraction_time'] > cutoff].reset_index(drop=True)

if hourly.empty:
    raise SystemExit('No rows returned: check the ticker, your licence, and the endpoint block.')

print(f"{len(hourly):,} rows | {hourly['topic_name'].nunique()} topics | "
      f"{hourly['extraction_time'].min():%Y-%m-%d %H:%M} → {hourly['extraction_time'].max():%Y-%m-%d %H:%M} UTC")

## Step 4: Inspect the schema

| Field | Meaning |
|---|---|
| `headline_count` | Distinct articles in the hour for this topic |
| `sentiment_sum` | Sum of scores (−1 to +1 each) |
| `sentiment_std` | Sample standard deviation within the hour (0 if fewer than two headlines) |

The main derived value is `sentiment_avg = sentiment_sum / headline_count`, in [−1, +1].

In [ ]:
columns = ['extraction_time', 'topic_name', 'sentiment_type', 'headline_count', 'sentiment_sum', 'sentiment_std']
sample = hourly[columns].sort_values('extraction_time').tail(8).copy()
sample['sentiment_avg'] = sample['sentiment_sum'] / sample['headline_count']
display(sample)

## Step 5: Daily sentiment by horizon

Topics are multi-label: one headline can sit in several topics. Adding rows across topics therefore gives a
**topic-weighted** average, and the count is a count of *topic-links*, not articles. That is the right input
for an overall reading, but label it accurately.

Hourly bars are grouped by the UTC day the headlines were received. A bar stamped 00:00 covers the last hour
of the previous day.

In [ ]:
hourly['date'] = receipt_date(hourly['extraction_time'])
daily = pool(hourly, ['date', 'sentiment_type'])

fig, (ax_sent, ax_vol) = plt.subplots(2, 1, figsize=(12, 7), sharex=True, gridspec_kw={'height_ratios': [3, 1]})
colours = {'short_term': '#1f77b4', 'medium_term': '#ff7f0e', 'long_term': '#2ca02c'}
for sentiment_type, group in daily.groupby('sentiment_type'):
    ax_sent.plot(group['date'], group['sentiment_avg'], marker='o', label=sentiment_type, color=colours.get(sentiment_type))
ax_sent.axhline(0, color='grey', lw=0.8, ls=':')
ax_sent.set_ylabel('Topic-weighted average')
ax_sent.set_title(f'{TICKER} | {INDEX_TYPE} | daily average sentiment by horizon')
ax_sent.legend()

volume = daily[daily['sentiment_type'] == SENTIMENT_TYPES[0]]
ax_vol.bar(volume['date'], volume['headline_count'], color='#7f8c8d')
ax_vol.set_ylabel('Topic-links')
plt.tight_layout()
plt.show()

## Step 6: Which topics carried the news this week?

Short-term horizon, ranked by volume. Averages from a handful of headlines are noisy, so treat small topics with care.

In [ ]:
by_topic = pool(hourly[hourly['sentiment_type'] == 'short_term'], ['topic_name'])
by_topic = by_topic.sort_values('headline_count', ascending=False)
display(by_topic[['topic_name', 'headline_count', 'sentiment_avg', 'sentiment_std']].head(10).round(3))

## Next steps

| Notebook | Use it to |
|---|---|
| `../live/gcsi_index_polling.ipynb` | Poll the hourly index continuously and build an indicator |
| `../live/gcsi_headline_polling.ipynb` | Work with individual headlines and match types |
| `../research/gcsi_bulk_history.ipynb` | Build a daily panel from 2015 from the bulk files |
| `../research/gcsi_topic_hierarchy.ipynb` | Break a move down by driver and topic |